# Data Validation

Checks the PDFs, the text extraction, and the labels of the IMWUT dataset before the
pipeline is trained, and reproduces every dataset figure reported in the Methodology
of the thesis. All reports are written to `daten/validation/`.

**Requirements**
- `01_cockpit.py` in the project folder (shared configuration with the pipeline notebook)
- label file `daten/labels/IMWUT_Dataset_original.csv`
- PDFs in `daten/pdfs/<year>/`; Markdown files in `daten/markdown/<year>/` (Sections 2, 8, 9)
- running Ollama server (Sections 7 and 8)

The notebook does not depend on variables from other notebooks and runs with
"Restart & Run All". Sections can also be run individually after the setup cells.

| Section | Thesis reference |
|---|---|
| 1 PDF integrity | Data Curation, *Validation* |
| 2 Markdown text quality | Document Preprocessing; Data Curation, *Cleaning* |
| 3 PDF text quality | Data Curation, *Validation*; Document Preprocessing |
| 4 Diagnosis of 2023_Dec_188 | Data Curation, *Cleaning* |
| 5 Label quality | Data Curation, *Validation* |
| 6 Dataset figures | Data Curation, *Binarization*; Dataset Statistics |
| 7 Embedding determinism | Pipeline A; Experimental Protocol |
| 8 Chunk size | Document Preprocessing |
| 9 Reference list removal | Document Preprocessing |
| 10 Folder inventory | no thesis statement; traceability of runs |

In [1]:
# Installs the pinned dependencies from requirements.txt.
# The file is created once from the working environment: pip freeze > requirements.txt
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Processing /builddir/build/BUILD/cockpit-367-build/cockpit-367/tmp/wheel/cockpit-367-py3-none-any.whl (from -r requirements.txt (line 19))
ERROR: Could not install packages due to an OSError: [Errno 2] Datei oder Verzeichnis nicht gefunden: '/builddir/build/BUILD/cockpit-367-build/cockpit-367/tmp/wheel/cockpit-367-py3-none-any.whl'

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Loads the shared configuration (paths, targets, label mapping, exclusions).
# -i runs the cockpit in the notebook namespace so that its functions see globals set later.
# The output documents the data basis: annotated (655), used (654), excluded papers,
# and the SHA-256 of the label file and the Classification Dataset.
%run -i 01_cockpit.py

from IPython.display import display

VALIDATION_ROOT = BASE_DIR / "daten" / "validation"
VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

/home/Mike-bazz/.local/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Projektstruktur wird initialisiert
Quelle: pdf_raw | Embedding: nomic-embed-text-v2-moe | LLM: llama3.2
Chunking: recursive_1200c_200o_prefixed_norefs | Pooling: mean | Seed: 42
Vorhersageziele: 9 Spalten

Daten-Audit
Label-Datei: IMWUT_Dataset_original.csv (SHA-256 ee5cf2727b11...)

ID-Nummerierung je Jahr
2023: 1..205, 205 Eintraege, lueckenlos
2024: 1..209, 208 Eintraege, bekannte Luecken [165]
2025: 1..243, 242 Eintraege, bekannte Luecken [173]
Ausgeschlossen: 2023_Dec_188 (Textfonts ohne Unicode-Zuordnung (Identity-H); Markdown zu 94,5 % U+FFFD, PDF-Text systematisch verschoben (Diagnose: Datenvalidierung, Abschnitt Diagnose); aus Tabelle entfernt)

Zusammenfassung Datenbasis
Theoretisches Maximum (ID-Zaehler): 657
Annotiert:                          655
Verwendet:                          654
Vorhandene PDFs (verwendet):        654
Fehlende PDFs:                      0
PDFs ohne Label-Eintrag:            0

Label-Mapping
code_required: 4 Eintraege umkodiert
data_required: 6 Eintr

## 1. PDF integrity

**Thesis:** Data Curation, paragraph *Validation* (first check: all 654 PDFs can be
opened, are not encrypted, and contain pages).

Checks for every paper of the Classification Dataset whether the PDF exists, can be
opened, is not encrypted, and has at least one page. Readable text is checked
separately in Sections 2 and 3.

Output: `pdf_report.csv` (status and reason per paper).

In [3]:
# Structural check of the PDF files; no text extraction.
# Status: ok | fehlt (missing) | leer (empty) | verschluesselt (encrypted) | beschaedigt (corrupt)
MIN_PDF_BYTES = 1024  # Kleinere Dateien sind praktisch leer

fitz.TOOLS.mupdf_display_errors(False)


def check_pdf(pdf_path: Path) -> tuple[str, str]:
    """Liefert (Status, Grund) fuer eine PDF."""
    if not pdf_path.exists():
        return "fehlt", "Datei nicht vorhanden"
    if pdf_path.stat().st_size < MIN_PDF_BYTES:
        return "leer", f"kleiner als {MIN_PDF_BYTES} Byte"
    try:
        with fitz.open(pdf_path) as doc:
            if doc.needs_pass:
                return "verschluesselt", "Passwort benoetigt"
            if doc.page_count == 0:
                return "leer", "0 Seiten"
            return "ok", f"{doc.page_count} Seiten"
    except Exception as e:
        return "beschaedigt", str(e)


pdf_report = []
for row in tqdm(master_label_df[["paper_id", "year"]].itertuples(index=False),
                total=len(master_label_df), desc="PDF-Check"):
    status, grund = check_pdf(STUDIEN_ROOT / row.year / f"{row.paper_id}.pdf")
    pdf_report.append({"paper_id": row.paper_id, "year": row.year, "status": status, "grund": grund})

pdf_report_df = pd.DataFrame(pdf_report)
pdf_report_df.to_csv(VALIDATION_ROOT / "pdf_report.csv", index=False)

print(pdf_report_df["status"].value_counts().to_string())
auffaellig = pdf_report_df[pdf_report_df["status"] != "ok"]
if not auffaellig.empty:
    display(auffaellig)

PDF-Check: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 654/654 [00:03<00:00, 200.60it/s]

status
ok    654


## 2. Markdown text quality

**Thesis:** Document Preprocessing: the Markdown files used by both pipelines contain
readable running text. Data Curation, *Cleaning*: no paper other than 2023_Dec_188 had
to be excluded.

Defective font encodings produce the replacement character U+FFFD. Only running text
is evaluated, i.e. lines without `<br>`: pymupdf4llm merges text from figures (axis and
diagram labels) into `<br>` lines, where U+FFFD does not affect the content. An earlier
criterion over the full text flagged 19 papers; the manual check of 2023_Dec_167,
2023_Mar_024, and 2025_Dec_240 showed readable running text (U+FFFD share in running
text: 0.006 %, 0 %, 0.13 %). The earlier criterion is still reported for comparison.

Thresholds are set values; the 1 % limit lies well above the manually checked papers.

Output: `markdown_report.csv`. Key columns: `ufffd_anteil_fliess` and
`zeilen_anteil_fliess` (criteria), `alt_kriterium_kritisch` (earlier criterion).
Skipped unless `DATEN_QUELLE == "markdown"`.

In [4]:
# Evaluates the Markdown files of all used papers.
# Criteria on running text (lines without <br>): share of U+FFFD and share of affected lines;
# additionally: near-empty conversions and many image placeholders with little text.
MIN_CHARS = 500                     # Darunter: praktisch kein Text extrahiert
MAX_UFFFD_ANTEIL_FLIESS = 0.01      # Anteil U+FFFD am Fliesstext (1 %)
MAX_ZEILEN_ANTEIL_FLIESS = 0.05     # Anteil Fliesstext-Zeilen mit mindestens einem U+FFFD (5 %)
MAX_BILDER_BEI_WENIG_TEXT = 8       # Viele Bild-Tags ...
MIN_WOERTER_BEI_BILDERN = 1500      # ... bei wenig Fliesstext

# Frueheres Kriterium ueber den gesamten Text; nur zum Vergleich ausgegeben
ALT_MAX_UFFFD_ABSOLUT = 150
ALT_MAX_UFFFD_ANTEIL = 0.02

# pymupdf4llm ersetzt Bilder ohne write_images durch Platzhalter wie
# "**==> picture [307 x 24] intentionally omitted <==**"; Markdown-Bildlinks zaehlen ebenfalls.
IMAGE_TAG = re.compile(r"==> picture \[[^\]]*\] intentionally omitted <==|!\[.*?\]\(.*?\)")
UFFFD = "\ufffd"


def bewerte_markdown(text: str) -> dict:
    zeilen = [z for z in text.splitlines() if z.strip()]
    fliess = [z for z in zeilen if "<br>" not in z]
    fliess_chars = sum(len(z) for z in fliess)

    n_ufffd = text.count(UFFFD)
    n_ufffd_fliess = sum(z.count(UFFFD) for z in fliess)
    betroffen_fliess = sum(UFFFD in z for z in fliess)
    anteil_fliess = n_ufffd_fliess / fliess_chars if fliess_chars else 0.0
    zeilen_anteil_fliess = betroffen_fliess / len(fliess) if fliess else 0.0

    chars, woerter = len(text), len(text.split())
    bilder = len(IMAGE_TAG.findall(text))

    if chars < MIN_CHARS:
        status = "defekt: kaum Text"
    elif anteil_fliess > MAX_UFFFD_ANTEIL_FLIESS or zeilen_anteil_fliess > MAX_ZEILEN_ANTEIL_FLIESS:
        status = "kritisch: Font-Encoding im Fliesstext"
    elif bilder > MAX_BILDER_BEI_WENIG_TEXT and woerter < MIN_WOERTER_BEI_BILDERN:
        status = "warnung: viele Bilder, wenig Text"
    else:
        status = "ok"

    alt_kritisch = n_ufffd > ALT_MAX_UFFFD_ABSOLUT or (chars and n_ufffd / chars > ALT_MAX_UFFFD_ANTEIL)
    return {"chars": chars, "woerter": woerter, "bild_tags": bilder,
            "n_ufffd_gesamt": n_ufffd, "n_ufffd_fliess": n_ufffd_fliess,
            "n_ufffd_br_zeilen": n_ufffd - n_ufffd_fliess,
            "fliess_zeilen": len(fliess), "fliess_zeilen_betroffen": betroffen_fliess,
            "ufffd_anteil_fliess": anteil_fliess, "zeilen_anteil_fliess": zeilen_anteil_fliess,
            "alt_kriterium_kritisch": bool(alt_kritisch), "status": status}


if DATEN_QUELLE == "markdown":
    md_report = []
    for row in master_label_df[["paper_id", "year"]].itertuples(index=False):
        md_path = MARKDOWN_ROOT / row.year / f"{row.paper_id}.md"
        eintrag = {"paper_id": row.paper_id, "year": row.year}
        if not md_path.exists():
            md_report.append({**eintrag, "status": "fehlt"})
            continue
        try:
            md_report.append({**eintrag, **bewerte_markdown(md_path.read_text(encoding="utf-8"))})
        except Exception as e:
            md_report.append({**eintrag, "status": f"lesefehler: {e}"})

    md_report_df = pd.DataFrame(md_report)
    md_report_df.to_csv(VALIDATION_ROOT / "markdown_report.csv", index=False)

    print(md_report_df["status"].value_counts().to_string())

    # Verteilung als Grundlage fuer die Wahl der Schwellen
    vorhanden = md_report_df.dropna(subset=["chars"])
    print("\nVerteilung U+FFFD-Anteil im Fliesstext:")
    print(vorhanden["ufffd_anteil_fliess"].describe(percentiles=[0.5, 0.9, 0.99]).to_string())
    print("\nVerteilung Anteil betroffener Fliesstext-Zeilen:")
    print(vorhanden["zeilen_anteil_fliess"].describe(percentiles=[0.5, 0.9, 0.99]).to_string())
    print(f"\nDokumente mit U+FFFD im Fliesstext: {(vorhanden['n_ufffd_fliess'] > 0).sum()}")
    print(f"Dokumente mit U+FFFD nur in <br>-Zeilen: "
          f"{((vorhanden['n_ufffd_gesamt'] > 0) & (vorhanden['n_ufffd_fliess'] == 0)).sum()}")

    # Vergleich mit dem frueheren Kriterium (fuer die Begruendung in der Arbeit)
    alt = vorhanden["alt_kriterium_kritisch"]
    neu = vorhanden["status"].str.startswith("kritisch")
    print(f"\nKritisch nach frueherem Kriterium (gesamter Text): {int(alt.sum())}")
    print(f"Kritisch nach neuem Kriterium (Fliesstext):       {int(neu.sum())}")
    print(f"Nur nach frueherem Kriterium kritisch:            {int((alt & ~neu).sum())}")

    spalten = ["paper_id", "status", "n_ufffd_gesamt", "n_ufffd_fliess",
               "ufffd_anteil_fliess", "zeilen_anteil_fliess", "alt_kriterium_kritisch"]
    auffaellig = md_report_df[(md_report_df["status"] != "ok") | md_report_df["alt_kriterium_kritisch"].fillna(False)]
    if not auffaellig.empty:
        with pd.option_context("display.max_rows", None):
            display(auffaellig[[s for s in spalten if s in auffaellig.columns]])
else:
    print(f"Uebersprungen (DATEN_QUELLE = '{DATEN_QUELLE}').")

status
ok                                       653
kritisch: Font-Encoding im Fliesstext      1

Verteilung U+FFFD-Anteil im Fliesstext:
count    654.000000
mean       0.000029
std        0.000089
min        0.000000
50%        0.000000
90%        0.000082
99%        0.000397
max        0.001269

Verteilung Anteil betroffener Fliesstext-Zeilen:
count    654.000000
mean       0.002696
std        0.006220
min        0.000000
50%        0.000000
90%        0.007472
99%        0.028929
max        0.067742

Dokumente mit U+FFFD im Fliesstext: 232
Dokumente mit U+FFFD nur in <br>-Zeilen: 32

Kritisch nach frueherem Kriterium (gesamter Text): 19
Kritisch nach neuem Kriterium (Fliesstext):       1
Nur nach frueherem Kriterium kritisch:            19


,paper_id,status,n_ufffd_gesamt,n_ufffd_fliess,ufffd_anteil_fliess,zeilen_anteil_fliess,alt_kriterium_kritisch
23,2023_Mar_024,ok,1663,0,0.000000,0.000000,True
48,2023_Jun_049,ok,1172,1,0.000011,0.003077,True
78,2023_Jun_079,ok,511,6,0.000070,0.005900,True
166,2023_Dec_167,ok,3404,6,0.000057,0.004926,True
233,2024_Mar_030,ok,678,0,0.000000,0.000000,True
340,2024_Sep_137,ok,610,0,0.000000,0.000000,True
346,2024_Sep_144,ok,704,1,0.000010,0.003236,True
350,2024_Sep_148,ok,264,0,0.000000,0.000000,True
359,2024_Dec_156,ok,219,0,0.000000,0.000000,True
366,2024_Dec_163,ok,1506,0,0.000000,0.000000,True


## 3. PDF text quality

**Thesis:** Data Curation, *Validation* (text extraction is referred to Document
Preprocessing); Document Preprocessing. Also covers the raw-PDF variant of the
pipeline (`DATEN_QUELLE == "pdf"`).

Extracts the text with PyMuPDF and evaluates lines of at least `MIN_ZEICHEN_ZEILE`
characters (shorter lines are mostly figure or table text). Defective encodings show
either as meaningless characters (U+FFFD, private-use area, control characters) or as
shifted characters with a low share of letters. Fonts without a Unicode mapping are
counted as a diagnosis only: they occur in 167 papers whose text is readable.

Result of the current run: all 654 papers pass; defective characters at most 0.04 %,
letters at least 86.4 % of the running text.

Output: `pdf_text_report.csv`. Key columns: `defekt_anteil_fliess`,
`buchstaben_anteil_fliess` (criteria), `fonts_ohne_unicode` (diagnosis).
Requires Section 2 (`UFFFD`).

In [5]:
# Evaluates the PyMuPDF text of all used PDFs.
# Criteria on running text: share of defective characters and share of letters.
# Font diagnosis (fonts without ToUnicode and without standard encoding) is not a criterion.
import unicodedata

MIN_PDF_CHARS = 500             # Darunter: praktisch kein Text extrahiert
MIN_ZEICHEN_ZEILE = 40          # Kuerzere Zeilen gelten als Abbildungs- oder Tabellentext
MAX_DEFEKT_ANTEIL = 0.01        # Anteil defekter Zeichen am Fliesstext (1 %)
MIN_BUCHSTABEN_ANTEIL = 0.6     # Anteil Buchstaben an den Nicht-Leerzeichen des Fliesstexts
STANDARD_ENCODINGS = {"WinAnsiEncoding", "MacRomanEncoding", "StandardEncoding"}

fitz.TOOLS.mupdf_display_errors(False)


def ist_defekt(c: str) -> bool:
    if c == UFFFD or 0xE000 <= ord(c) <= 0xF8FF:
        return True
    return unicodedata.category(c) == "Cc" and c not in "\n\r\t"


def font_diagnose(doc) -> dict:
    """Zaehlt Fonts ohne ToUnicode-Zuordnung und ohne Standard-Encoding (je Dokument eindeutig)."""
    fonts = {}
    for seite in doc:
        for xref, _ext, typ, basefont, _name, encoding, *_ in seite.get_fonts():
            if xref and xref not in fonts:
                hat_unicode = doc.xref_get_key(xref, "ToUnicode")[0] != "null"
                fonts[xref] = (typ, basefont, hat_unicode, encoding)
    ohne = [f for f in fonts.values() if not f[2] and f[3] not in STANDARD_ENCODINGS]
    return {"fonts": len(fonts), "fonts_ohne_unicode": len(ohne),
            "type3_fonts": sum(f[0] == "Type3" for f in fonts.values())}


def bewerte_pdf_text(pdf_path: Path) -> dict:
    with fitz.open(pdf_path) as doc:
        text = "\n".join(seite.get_text("text") for seite in doc)
        diagnose = font_diagnose(doc)

    zeilen = [z.strip() for z in text.splitlines() if z.strip()]
    fliess = [z for z in zeilen if len(z) >= MIN_ZEICHEN_ZEILE]
    fliess_text = "".join(fliess)
    nicht_leer = [c for c in fliess_text if not c.isspace()]
    n = len(nicht_leer)

    n_defekt = sum(ist_defekt(c) for c in nicht_leer)
    defekt_anteil = n_defekt / n if n else 0.0
    buchstaben_anteil = sum(c.isalpha() for c in nicht_leer) / n if n else 0.0

    if len(text) < MIN_PDF_CHARS or n == 0:
        status = "defekt: kaum Text"
    elif defekt_anteil > MAX_DEFEKT_ANTEIL:
        status = "kritisch: defekte Zeichen"
    elif buchstaben_anteil < MIN_BUCHSTABEN_ANTEIL:
        status = "kritisch: wenig Buchstaben"
    else:
        status = "ok"

    return {"chars": len(text), "fliess_zeilen": len(fliess), "n_defekt_fliess": n_defekt,
            "defekt_anteil_fliess": defekt_anteil, "buchstaben_anteil_fliess": buchstaben_anteil,
            **diagnose, "status": status}


pdf_text_report = []
for row in tqdm(master_label_df[["paper_id", "year"]].itertuples(index=False),
                total=len(master_label_df), desc="PDF-Text"):
    pdf_path = STUDIEN_ROOT / row.year / f"{row.paper_id}.pdf"
    eintrag = {"paper_id": row.paper_id, "year": row.year}
    if not pdf_path.exists():
        pdf_text_report.append({**eintrag, "status": "fehlt"})
        continue
    try:
        pdf_text_report.append({**eintrag, **bewerte_pdf_text(pdf_path)})
    except Exception as e:
        pdf_text_report.append({**eintrag, "status": f"lesefehler: {e}"})

pdf_text_df = pd.DataFrame(pdf_text_report)
pdf_text_df.to_csv(VALIDATION_ROOT / "pdf_text_report.csv", index=False)

print(pdf_text_df["status"].value_counts().to_string())
vorhanden = pdf_text_df.dropna(subset=["chars"])
print("\nVerteilung Anteil defekter Zeichen im Fliesstext:")
print(vorhanden["defekt_anteil_fliess"].describe(percentiles=[0.5, 0.9, 0.99]).to_string())
print("\nVerteilung Buchstabenanteil im Fliesstext:")
print(vorhanden["buchstaben_anteil_fliess"].describe(percentiles=[0.01, 0.1, 0.5]).to_string())
print(f"\nDokumente mit Fonts ohne Unicode-Zuordnung: {(vorhanden['fonts_ohne_unicode'] > 0).sum()}")

auffaellig = pdf_text_df[pdf_text_df["status"] != "ok"]
if not auffaellig.empty:
    with pd.option_context("display.max_rows", None):
        display(auffaellig)

PDF-Text: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 654/654 [02:18<00:00,  4.72it/s]

status
ok    654

Verteilung Anteil defekter Zeichen im Fliesstext:
count    654.000000
mean       0.000013
std        0.000033
min        0.000000
50%        0.000000
90%        0.000029
99%        0.000137
max        0.000402

Verteilung Buchstabenanteil im Fliesstext:
count    654.000000
mean       0.918162
std        0.012515
min        0.864032
1%         0.881214
10%        0.902647
50%        0.919930
max        0.948419

Dokumente mit Fonts ohne Unicode-Zuordnung: 167


## 4. Diagnosis of the excluded paper 2023_Dec_188

**Thesis:** Data Curation, *Cleaning* (reason for excluding 2023_Dec_188).

The paper is listed in `EXCLUDED_PAPER_IDS` and is not processed by the pipeline.
This cell evaluates its PDF text (Section 3), lists the fonts of page 1, converts only
this PDF with the same call as the pipeline, and evaluates the result (Section 2).
The Markdown is stored in `daten/validation/diagnose/`, so the pipeline caches remain
untouched.

Result: the text fonts are embedded as Type0 / Identity-H without Unicode mapping.
The PDF text consists of characters shifted by a constant offset; in the Markdown,
94.5 % of the running text are U+FFFD. Decision: the paper remains excluded.

Requires Sections 2 and 3.

In [6]:
# Diagnosis of a single excluded paper: PDF text, fonts of page 1, Markdown conversion
# (same call as the pipeline, stored outside MARKDOWN_ROOT), sample of the running text.
import pymupdf4llm

PAPER = "2023_Dec_188"
pdf_path = STUDIEN_ROOT / PAPER[:4] / f"{PAPER}.pdf"
if not pdf_path.exists():
    raise FileNotFoundError(f"PDF fehlt: {pdf_path} (ggf. manuell als {PAPER}.pdf ablegen)")

diagnose_dir = VALIDATION_ROOT / "diagnose"
diagnose_dir.mkdir(parents=True, exist_ok=True)

# 1. PDF-Text und Fonts
pdf_bewertung = bewerte_pdf_text(pdf_path)
print("PDF-Text:")
for k, v in pdf_bewertung.items():
    print(f"  {k:<26} {v:.4f}" if isinstance(v, float) else f"  {k:<26} {v}")

with fitz.open(pdf_path) as doc:
    print("\nFonts auf Seite 1:")
    for xref, _ext, typ, basefont, _name, encoding, *_ in doc[0].get_fonts():
        hat_unicode = doc.xref_get_key(xref, "ToUnicode")[0] != "null" if xref else None
        print(f"  {basefont:<40} {typ:<10} Encoding: {encoding or '-':<18} ToUnicode: {hat_unicode}")
    seite1 = doc[0].get_text("text")
print("\nAuszug PDF-Text, Seite 1:")
print(repr(seite1[:400]))

# 2. Nur diese Studie nach Markdown konvertieren (gleicher Aufruf wie in der Pipeline)
md_text = pymupdf4llm.to_markdown(str(pdf_path))
md_path = diagnose_dir / f"{PAPER}.md"
md_path.write_text(md_text, encoding="utf-8")
print(f"\nMarkdown gespeichert: {md_path}")

md_bewertung = bewerte_markdown(md_text)
print("\nMarkdown:")
for k, v in md_bewertung.items():
    print(f"  {k:<26} {v:.4f}" if isinstance(v, float) else f"  {k:<26} {v}")

# 3. Stichprobe des Fliesstexts zur manuellen Sichtung
fliess = [z for z in md_text.splitlines() if z.strip() and "<br>" not in z]
print("\nErste Fliesstext-Zeilen:")
for z in fliess[:8]:
    print("  ", z[:160])
betroffen = [z for z in fliess if UFFFD in z]
print(f"\nFliesstext-Zeilen mit U+FFFD: {len(betroffen)} von {len(fliess)}")
for z in betroffen[:5]:
    print("  ", z[:160])

PDF-Text:
  chars                      84236
  fliess_zeilen              765
  n_defekt_fliess            2242
  defekt_anteil_fliess       0.0451
  buchstaben_anteil_fliess   0.8957
  fonts                      25
  fonts_ohne_unicode         12
  type3_fonts                0
  status                     kritisch: defekte Zeichen

Fonts auf Seite 1:
  JNIEMP+txsys                             Type1      Encoding: -                  ToUnicode: True
  LinBiolinumOB-Identity-H-Identity-H      Type0      Encoding: Identity-H         ToUnicode: False
  LinBiolinumO-Identity-H-Identity-H       Type0      Encoding: Identity-H         ToUnicode: False
  LinLibertineO-Identity-H-Identity-H      Type0      Encoding: Identity-H         ToUnicode: False
  LinLibertineOB-Identity-H-Identity-H     Type0      Encoding: Identity-H         ToUnicode: False
  LinLibertineOI-Identity-H-Identity-H     Type0      Encoding: Identity-H         ToUnicode: False

Auszug PDF-Text, Seite 1:
'\x12\x19\x19\n54\x1

## 5. Label quality

**Thesis:** Data Curation, paragraph *Validation* (second and third check).

Checks the raw labels (`master_label_raw_df`, before binarization):
- **A** every label field against the allowed values of the annotation schema
  (Table `tab:m_criteria`),
- **B** `paper_type`: formatting and unknown categories,
- **C** consistency of requirement, availability, and link fields of code and data.

Result of the current run: A and B without findings. C reports only links without an
available artifact, available artifacts without a link (both permitted by the schema),
and one paper (2025_Mar_011) with available data although `data_required` is Unknown.
Counts in C refer to raw values (`Yes` only); the figures in the thesis refer to the
binarized targets (Section 6, cell G).

Output: `label_report.csv` (check and paper_id per finding).

In [7]:
# Validates the raw labels: allowed values (A), paper_type format (B), cross-field consistency (C).
# NaN in the label file stands for N/A.
ERLAUBTE_WERTE = {
    "is_dataset_paper": {"Yes", "No"},
    "open_access": {"Yes", "No", "Unknown"},
    "code_required": {"Yes", "No", "Unknown"},
    "data_required": {"Yes", "No", "Unknown"},
    "hardware_required": {"Yes", "No", "Unknown", "N/A"},
    "software_required": {"Yes", "No", "Unknown", "N/A"},
    "code_available": {"Yes", "No", "Partially", "Upon_request", "Unknown", "N/A"},
    "data_available": {"Yes", "No", "Partially", "Upon_request", "Unknown", "N/A"},
    "instructions_present": {"Yes", "No", "Partially", "Unknown", "N/A"},
}

raw = master_label_raw_df.copy()
befunde = []


def melde(pruefung: str, maske: pd.Series, detail: str = ""):
    for p_id in raw.loc[maske, "paper_id"]:
        befunde.append({"pruefung": pruefung, "paper_id": p_id, "detail": detail})


# A: Werte ausserhalb des Codebooks
for col, erlaubt in ERLAUBTE_WERTE.items():
    werte = raw[col].fillna("N/A").astype(str)
    melde(f"{col}: unzulaessiger Wert", ~werte.isin(erlaubt))

# B: paper_type - Formatierung und Kategorien
pt = raw["paper_type"].fillna("").astype(str)
melde("paper_type: leer", pt.str.strip() == "")
melde("paper_type: Leerzeichen am Rand", pt != pt.str.strip())
melde("paper_type: Leerzeichen am Semikolon", pt.str.contains(r"\s;|;\s", regex=True))
melde("paper_type: doppeltes Leerzeichen", pt.str.contains("  ", regex=False))
kategorien = pt.str.split(";").apply(lambda xs: {x.strip() for x in xs if x.strip()})
melde("paper_type: unbekannte Kategorie", kategorien.apply(lambda k: bool(k - set(PAPER_TYPE_CATEGORIES))))

# C: Logische Konsistenz zwischen Kriterien
for art in ["code", "data"]:
    req, avail, url = raw[f"{art}_required"], raw[f"{art}_available"], raw[f"{art}_url"]
    melde(f"{art}: nicht benoetigt, aber verfuegbar", req.isin(["No", "Unknown"]) & avail.eq("Yes"))
    melde(f"{art}: benoetigt, aber available = N/A", req.eq("Yes") & avail.isna())
    melde(f"{art}: verfuegbar, aber keine URL", avail.eq("Yes") & url.isna())
    melde(f"{art}: URL, aber nicht verfuegbar", avail.isin(["No", "Unknown"]) & url.notna(),
          "erwartbar bei leeren oder nur README enthaltenden Repositories")

befunde_df = pd.DataFrame(befunde, columns=["pruefung", "paper_id", "detail"])
befunde_df.to_csv(VALIDATION_ROOT / "label_report.csv", index=False)

if befunde_df.empty:
    print(f"Keine Befunde in {len(raw)} Zeilen.")
else:
    print(befunde_df["pruefung"].value_counts().to_string())

pruefung
data: verfuegbar, aber keine URL          45
code: URL, aber nicht verfuegbar          19
data: URL, aber nicht verfuegbar          14
code: verfuegbar, aber keine URL           2
data: nicht benoetigt, aber verfuegbar     1


## 6. Dataset figures reported in the Methodology

Reproduces the dataset figures of the thesis. Each cell names the paragraph it supports.

| Cell | Thesis reference | Reproduces |
|---|---|---|
| D | Data Curation, *Binarization* | N/A in availability fields only if the requirement is No; 77 of 85 (code), 123 of 179 (data) |
| E | Dataset Statistics, Table `tab:m_distribution`, Figure `fig:m_paper_types`a | positive labels per target |
| F | Dataset Statistics, *Paper types*; Figure `fig:m_paper_types`b | types per paper, mean 2.3, combinations, 61 convention-labeled papers |
| G | Dataset Statistics, *Artifact-related targets* | availability among required papers; links without availability (20, 18); availability without links (3, 72) |

All cells except D work on the binarized Classification Dataset (`master_label_df`).

In [8]:
# D: N/A in the availability fields versus the requirement fields (raw labels).
# Thesis: Binarization. Checks that N/A occurs only if *_required = No and counts these cases.
# Output: availability_na_report.csv
raw_na = master_label_raw_df.copy()
for col in ["code_required", "code_available", "data_required", "data_available"]:
    raw_na[col] = raw_na[col].fillna("N/A").astype(str)

na_zeilen = []
for art in ["code", "data"]:
    req, avail = raw_na[f"{art}_required"], raw_na[f"{art}_available"]

    # Vollstaendige Kreuztabelle zur Einsicht
    print(f"\n{art}_required x {art}_available")
    display(pd.crosstab(req, avail, margins=True, margins_name="Summe"))

    nicht_benoetigt = req.eq("No")
    na_gesamt = avail.eq("N/A")
    na_bei_no = nicht_benoetigt & na_gesamt

    # Pruefung 1: N/A nur bei Requirement = No
    na_ausserhalb = raw_na.loc[na_gesamt & ~nicht_benoetigt, "paper_id"].tolist()

    # Pruefung 2: Anteil N/A unter den nicht benoetigten Artefakten
    uebrige = avail[nicht_benoetigt & ~na_gesamt].value_counts().to_dict()

    na_zeilen.append({
        "artefakt": art,
        "required_no": int(nicht_benoetigt.sum()),
        "available_na_bei_required_no": int(na_bei_no.sum()),
        "available_na_gesamt": int(na_gesamt.sum()),
        "na_ausserhalb_required_no": len(na_ausserhalb),
        "uebrige_werte_bei_required_no": uebrige,
    })
    if na_ausserhalb:
        print(f"WARNUNG {art}: N/A ohne required = No bei {na_ausserhalb}")

na_report_df = pd.DataFrame(na_zeilen)
na_report_df.to_csv(VALIDATION_ROOT / "availability_na_report.csv", index=False)
display(na_report_df)

for z in na_zeilen:
    print(f"{z['artefakt']}: {z['available_na_bei_required_no']} von {z['required_no']} Papers "
          f"mit {z['artefakt']}_required = No haben {z['artefakt']}_available = N/A; "
          f"N/A ausserhalb von required = No: {z['na_ausserhalb_required_no']}")


code_required x code_available


code_available,N/A,No,Partially,Unknown,Upon_request,Yes,Summe
code_required,,,,,,,
No,77,7,0,0,1,0,85
Unknown,0,4,0,0,0,0,4
Yes,0,409,4,1,1,150,565
Summe,77,420,4,1,2,150,654



data_required x data_available


data_available,N/A,No,Partially,Unknown,Upon_request,Yes,Summe
data_required,,,,,,,
No,123,55,0,0,1,0,179
Unknown,0,3,0,2,0,1,6
Yes,0,271,50,6,5,137,469
Summe,123,329,50,8,6,138,654


,artefakt,required_no,available_na_bei_required_no,available_na_gesamt,na_ausserhalb_required_no,uebrige_werte_bei_required_no
0,code,85,77,77,0,"{'No': 7, 'Upon_request': 1}"
1,data,179,123,123,0,"{'No': 55, 'Upon_request': 1}"


code: 77 von 85 Papers mit code_required = No haben code_available = N/A; N/A ausserhalb von required = No: 0
data: 123 von 179 Papers mit data_required = No haben data_available = N/A; N/A ausserhalb von required = No: 0


In [9]:
# E: Positive labels per target compared with the values reported in the thesis.
# Thesis: Table tab:m_distribution (artifact targets) and Figure fig:m_paper_types a (paper types).
# Output: tabelle_positive_labels.csv
TABELLE_METHODOLOGY = {
    "Algorithm Development": 498, "User Study": 389, "Software Development": 351,
    "Hardware Development": 174, "Dataset Paper": 57, "Literature Review": 12,
    "code_required": 565, "data_required": 469, "hardware_required": 419,
    "software_required": 243, "data_available": 188, "code_url": 171,
    "code_available": 154, "data_url": 134,
}

df = master_label_df
n = len(df)
typen = df["paper_type"].fillna("").astype(str).str.split(";").apply(lambda xs: {x.strip() for x in xs})

zeilen = []
for ziel, erwartet in TABELLE_METHODOLOGY.items():
    if ziel in PAPER_TYPE_CATEGORIES:
        ist = int(typen.apply(lambda t: ziel in t).sum())
    else:
        ist = int(df[ziel].eq("Yes").sum())
    zeilen.append({"ziel": ziel, "tabelle": erwartet, "daten": ist,
                   "anteil_daten": f"{ist / n:.1%}", "stimmt": ist == erwartet})

vergleich_df = pd.DataFrame(zeilen)
vergleich_df.to_csv(VALIDATION_ROOT / "tabelle_positive_labels.csv", index=False)
display(vergleich_df)

abweichend = vergleich_df[~vergleich_df["stimmt"]]
print("Tabelle stimmt mit den Daten ueberein." if abweichend.empty
      else f"Abweichungen bei: {abweichend['ziel'].tolist()}")

,ziel,tabelle,daten,anteil_daten,stimmt
0,Algorithm Development,498,498,76.1%,True
1,User Study,389,389,59.5%,True
2,Software Development,351,351,53.7%,True
3,Hardware Development,174,174,26.6%,True
4,Dataset Paper,57,57,8.7%,True
5,Literature Review,12,12,1.8%,True
6,code_required,565,565,86.4%,True
7,data_required,469,469,71.7%,True
8,hardware_required,419,419,64.1%,True
9,software_required,243,243,37.2%,True


Tabelle stimmt mit den Daten ueberein.


In [10]:
# F: Combinations of paper types (all 63 possible combinations, abbreviations AD, HD, SD, US, DP, LR).
# Thesis: Dataset Statistics, paragraph "Paper types"; Figure fig:m_paper_types b.
# Output: paper_type_kombinationen.csv
from itertools import combinations

# Kuerzel je Paper-Typ; Reihenfolge wie PAPER_TYPE_CATEGORIES
KUERZEL = {
    "Algorithm Development": "AD",
    "Hardware Development": "HD",
    "Software Development": "SD",
    "User Study": "US",
    "Dataset Paper": "DP",
    "Literature Review": "LR",
}
assert set(KUERZEL) == set(PAPER_TYPE_CATEGORIES), "Kuerzel passen nicht zu PAPER_TYPE_CATEGORIES"

print("Legende:")
for typ, kz in KUERZEL.items():
    print(f"  {kz} = {typ}")

df = master_label_df
n = len(df)

# Typen je Paper als Tupel in der festen Reihenfolge von PAPER_TYPE_CATEGORIES,
# damit "User Study;Algorithm Development" und "Algorithm Development;User Study"
# als dieselbe Kombination gezaehlt werden.
def typen_tupel(wert) -> tuple:
    typen = {t.strip() for t in str(wert).split(";") if t.strip()} if pd.notna(wert) else set()
    unbekannt = typen - set(PAPER_TYPE_CATEGORIES)
    if unbekannt:
        raise ValueError(f"Unbekannte Paper-Typen: {unbekannt}")
    return tuple(t for t in PAPER_TYPE_CATEGORIES if t in typen)

def als_kuerzel(kombi: tuple) -> str:
    return "+".join(KUERZEL[t] for t in kombi)

kombi_je_paper = df["paper_type"].apply(typen_tupel)

# Kontrolle: jedes Paper hat mindestens einen Typ
leer = df.loc[kombi_je_paper.str.len() == 0, "paper_id"].tolist()
print(f"\nPapers ohne Paper-Typ: {len(leer)} {leer if leer else ''}")

# Alle moeglichen Kombinationen (Groesse 1 bis 6) mit ihrer Haeufigkeit
haeufigkeit = kombi_je_paper.value_counts()
zeilen = []
for k in range(1, len(PAPER_TYPE_CATEGORIES) + 1):
    for kombi in combinations(PAPER_TYPE_CATEGORIES, k):
        anzahl = int(haeufigkeit.get(kombi, 0))
        zeilen.append({"anzahl_typen": k, "kombination": als_kuerzel(kombi),
                       "papers": anzahl, "anteil": f"{anzahl / n:.1%}"})

# Sortierung: zuerst nach Anzahl der Typen (aufsteigend),
# innerhalb gleicher Anzahl nach Haeufigkeit (absteigend)
kombi_df = (pd.DataFrame(zeilen)
            .sort_values(["anzahl_typen", "papers"], ascending=[True, False])
            .reset_index(drop=True))
kombi_df.to_csv(VALIDATION_ROOT / "paper_type_kombinationen.csv", index=False)

# Kontrolle: Summe ueber alle Kombinationen muss der Zahl der Papers entsprechen
assert kombi_df["papers"].sum() == n, "Kombinationen decken nicht alle Papers ab"

print(f"\nMoegliche Kombinationen: {len(kombi_df)} | im Datensatz vorhanden: "
      f"{(kombi_df['papers'] > 0).sum()}")
print("\nVorhandene Kombinationen:")
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(kombi_df[kombi_df["papers"] > 0])

# Verteilung der Anzahl Typen je Paper (Abbildung fig:m_paper_types b)
anzahl_typen = kombi_je_paper.str.len()
print("\nAnzahl Typen je Paper:")
print(anzahl_typen.value_counts().sort_index().to_string())

# Mittelwert auf zwei Wegen: direkt und ueber die Summe der Typ-Zuweisungen
# (entspricht der Summe der Balken in Abbildung fig:m_paper_types a)
zuweisungen = int(anzahl_typen.sum())
print(f"\nTyp-Zuweisungen gesamt: {zuweisungen}")
print(f"Mittlere Anzahl Typen je Paper: {zuweisungen} / {n} = {zuweisungen / n:.3f}")

# Papers mit genau einem Typ (fuer den Satz zu den Single-Type-Papers)
print("\nPapers mit genau einem Typ:")
print(kombi_df[kombi_df["anzahl_typen"] == 1][["kombination", "papers"]].to_string(index=False))

Legende:
  AD = Algorithm Development
  HD = Hardware Development
  SD = Software Development
  US = User Study
  DP = Dataset Paper
  LR = Literature Review

Papers ohne Paper-Typ: 0 

Moegliche Kombinationen: 63 | im Datensatz vorhanden: 30

Vorhandene Kombinationen:


,anzahl_typen,kombination,papers,anteil
0,1,AD,63,9.6%
1,1,US,50,7.6%
2,1,SD,9,1.4%
3,1,LR,9,1.4%
4,1,DP,6,0.9%
5,1,HD,4,0.6%
6,2,AD+SD,86,13.1%
7,2,AD+US,63,9.6%
8,2,SD+US,34,5.2%
9,2,AD+HD,33,5.0%



Anzahl Typen je Paper:
paper_type
1    141
2    259
3    197
4     54
5      3

Typ-Zuweisungen gesamt: 1481
Mittlere Anzahl Typen je Paper: 1481 / 654 = 2.265

Papers mit genau einem Typ:
kombination  papers
         AD      63
         US      50
         SD       9
         LR       9
         DP       6
         HD       4


In [11]:
# G: Requirement, availability, and link after binarization (not raw values as in Section 5 C).
# Thesis: Dataset Statistics, paragraph "Artifact-related targets" and the following paragraph.
# Output: artefakt_beziehungen.csv
df = master_label_df
zeilen = []
for art in ["code", "data"]:
    req, avail, url = (df[f"{art}_{f}"].eq("Yes") for f in ["required", "available", "url"])
    zeilen.append({
        "artefakt": art,
        "required": int(req.sum()),
        "available_und_required": int((avail & req).sum()),
        "anteil_available_unter_required": f"{(avail & req).sum() / req.sum():.1%}",
        "url_ohne_available": int((url & ~avail).sum()),
        "available_ohne_url": int((avail & ~url).sum()),
    })

beziehungen_df = pd.DataFrame(zeilen)
beziehungen_df.to_csv(VALIDATION_ROOT / "artefakt_beziehungen.csv", index=False)
display(beziehungen_df)

,artefakt,required,available_und_required,anteil_available_unter_required,url_ohne_available,available_ohne_url
0,code,565,154,27.3%,20,3
1,data,469,187,39.9%,18,72


## 7. Embedding determinism

**Thesis:** Pipeline A; Experimental Protocol (reproducibility of the embeddings).

Embeds two randomly drawn papers (seed from the cockpit) twice in a row (comparison A)
and compares the vectors with the stored embeddings from Segment 3 of the pipeline
notebook (comparison B, different session). Embedding uses the same function as the
pipeline, including the splitting of overly long chunks.

Requires a running Ollama server; comparison B requires stored embeddings.

In [12]:
# Compares repeated embeddings of the same text: bit-identity, max. absolute deviation, cosine distance.
N_DOKUMENTE = 2


def vergleiche(a: list, b: list) -> dict:
    """Bitgleichheit, groesste absolute Abweichung und Kosinus-Distanz (1 - cos) zweier Vektoren."""
    va, vb = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    if va.shape != vb.shape:
        return {"bitgleich": False, "max_abs_diff": np.nan, "cos_distanz": np.nan}
    if np.array_equal(va, vb):
        return {"bitgleich": True, "max_abs_diff": 0.0, "cos_distanz": 0.0}
    return {
        "bitgleich": False,
        "max_abs_diff": float(np.max(np.abs(va - vb))),
        "cos_distanz": float(1 - va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb))),
    }


def vektorisiere(chunks: list, beschreibung: str) -> list:
    """Liste aller (Teil-)Chunks als {"text", "vector"}; None-Eintraege bei Fehlern."""
    teile = []
    for c in tqdm(chunks, desc=beschreibung):
        ergebnis = embed_chunk(c["text"])
        teile += ergebnis if ergebnis is not None else [{"text": c["text"], "vector": None}]
    return teile


if not check_ollama():
    print("Uebersprungen: Ollama-Server nicht erreichbar.")
else:
    quelle_dir, endung = (MARKDOWN_ROOT, ".md") if DATEN_QUELLE == "markdown" else (STUDIEN_ROOT, ".pdf")
    stichprobe = master_label_df[["paper_id", "year"]].sample(n=N_DOKUMENTE, random_state=GLOBAL_SEED)

    ergebnisse = []
    for row in stichprobe.itertuples(index=False):
        quelle = quelle_dir / row.year / f"{row.paper_id}{endung}"
        if not quelle.exists():
            print(f"{row.paper_id}: Quelldatei fehlt, uebersprungen")
            continue

        chunks, _ = extract_pdf_structure(quelle)
        lauf_1 = vektorisiere(chunks, f"{row.paper_id} Lauf 1")
        lauf_2 = vektorisiere(chunks, f"{row.paper_id} Lauf 2")

        # Gespeicherte Vektoren aus Segment 3 (andere Sitzung), zugeordnet ueber den Text
        gespeichert = {}
        emb_path = CURRENT_EMBEDDING_PATH / row.year / f"{row.paper_id}_embeddings.json"
        if emb_path.exists():
            daten = json.loads(emb_path.read_text(encoding="utf-8"))
            gespeichert = {c["text"]: c["vector"] for c in daten["chunks"]}

        if [t["text"] for t in lauf_1] != [t["text"] for t in lauf_2]:
            print(f"{row.paper_id}: Nachteilen lieferte in beiden Laeufen unterschiedliche Teile")
            continue

        for nr, (t1, t2) in enumerate(zip(lauf_1, lauf_2)):
            eintrag = {"paper_id": row.paper_id, "teil": nr, "chars": len(t1["text"])}
            if t1["vector"] is None or t2["vector"] is None:
                ergebnisse.append({**eintrag, "vergleich": "A: Lauf 1 vs. Lauf 2", "fehler": "kein Vektor"})
                continue
            ergebnisse.append({**eintrag, "vergleich": "A: Lauf 1 vs. Lauf 2", **vergleiche(t1["vector"], t2["vector"])})

            alt = gespeichert.get(t1["text"])
            if gespeichert and alt is None:
                ergebnisse.append({**eintrag, "vergleich": "B: gespeichert vs. Lauf 1",
                                   "fehler": "Text nicht in gespeicherten Chunks"})
            elif alt is not None:
                ergebnisse.append({**eintrag, "vergleich": "B: gespeichert vs. Lauf 1",
                                   **vergleiche(alt, t1["vector"])})

    det_df = pd.DataFrame(ergebnisse)
    det_df.to_csv(VALIDATION_ROOT / "embedding_determinismus.csv", index=False)

    print(f"\nStichprobe: {', '.join(stichprobe['paper_id'])}")
    if det_df.empty:
        print("Keine Vergleiche durchgefuehrt.")
    else:
        if "fehler" in det_df:
            fehler = det_df[det_df["fehler"].notna()]
            if not fehler.empty:
                print(fehler.groupby(["vergleich", "fehler"]).size().to_string())
            det_df = det_df[det_df["fehler"].isna()]
        det_df = det_df.astype({"bitgleich": bool})
        zusammenfassung = det_df.groupby("vergleich").agg(
            teile=("bitgleich", "size"),
            bitgleich=("bitgleich", "sum"),
            max_abs_diff=("max_abs_diff", "max"),
            max_cos_distanz=("cos_distanz", "max"),
        )
        print(zusammenfassung.to_string(float_format=lambda x: f"{x:.3e}"))
        if "B: gespeichert vs. Lauf 1" not in zusammenfassung.index:
            print("Vergleich B nicht moeglich: keine passenden gespeicherten Embeddings.")

2023_Jun_079 Lauf 2: 100%|████████████████████████████████████████████████████████████████████████████████████████| 84/84 [00:10<00:00,  7.95it/s]


Stichprobe: 2024_Sep_130, 2023_Jun_079
                           teile  bitgleich  max_abs_diff  max_cos_distanz
vergleich                                                                 
A: Lauf 1 vs. Lauf 2         193        193     0.000e+00        0.000e+00
B: gespeichert vs. Lauf 1    193        193     0.000e+00        0.000e+00


## 8. Chunk size and context window

**Thesis:** Document Preprocessing (justification of `CHUNK_SIZE`).

Measures for several chunk sizes how many chunks exceed the context window of the
embedding model. Characters and tokens have no fixed ratio, so the token counts are
taken directly from Ollama. No vectors are stored.

Runtime: roughly documents x chunks per document x number of sizes Ollama calls.
Requires a running Ollama server and `DATEN_QUELLE == "markdown"`.

In [13]:
# Chunks a sample of papers with each size in MESS_GROESSEN (same splitter and overlap as the
# cockpit) and counts the chunks that exceed the context window of the embedding model.
MESS_GROESSEN = [1200, 800, 600]   # Zeichen
N_MESS_DOKUMENTE = 10


def chunks_fuer_groesse(text: str, groesse: int) -> list:
    """Chunking wie im Cockpit, nur mit anderer Groesse (Overlap unveraendert)."""
    return RecursiveCharacterTextSplitter(
        chunk_size=groesse,
        chunk_overlap=min(CHUNK_OVERLAP, groesse // 2),
        separators=CHUNK_SEPARATORS,
    ).split_text(text)


if not check_ollama():
    print("Uebersprungen: Ollama-Server nicht erreichbar.")
elif DATEN_QUELLE != "markdown":
    print(f"Uebersprungen (DATEN_QUELLE = '{DATEN_QUELLE}').")
else:
    mess_stichprobe = master_label_df[["paper_id", "year"]].sample(n=N_MESS_DOKUMENTE, random_state=GLOBAL_SEED)
    texte = {
        row.paper_id: (MARKDOWN_ROOT / row.year / f"{row.paper_id}.md").read_text(encoding="utf-8")
        for row in mess_stichprobe.itertuples(index=False)
        if (MARKDOWN_ROOT / row.year / f"{row.paper_id}.md").exists()
    }
    n_aufrufe = sum(len(chunks_fuer_groesse(t, g)) for t in texte.values() for g in MESS_GROESSEN)
    print(f"Stichprobe: {len(texte)} Dokumente, {n_aufrufe} Ollama-Aufrufe")

    messung = []
    for groesse in MESS_GROESSEN:
        for p_id, text in tqdm(texte.items(), desc=f"{groesse} Zeichen"):
            for nr, chunk in enumerate(chunks_fuer_groesse(text, groesse)):
                try:
                    status = "ok" if get_embedding(chunk, raise_on_context=True) else "fehler"
                except KontextZuLangError:
                    status = "zu_lang"
                messung.append({"groesse": groesse, "paper_id": p_id, "chunk": nr,
                                "chars": len(chunk), "status": status,
                                "anfang": chunk[:120] if status == "zu_lang" else ""})

    mess_df = pd.DataFrame(messung)
    mess_df.to_csv(VALIDATION_ROOT / "chunk_kontextfenster.csv", index=False)

    zu_lang = mess_df["status"].eq("zu_lang")
    uebersicht = mess_df.assign(zu_lang=zu_lang).groupby("groesse").agg(
        chunks=("status", "size"),
        zu_lang=("zu_lang", "sum"),
        fehler=("status", lambda s: (s == "fehler").sum()),
    )
    uebersicht["anteil_zu_lang"] = uebersicht["zu_lang"] / uebersicht["chunks"]
    uebersicht["dokumente_betroffen"] = (
        mess_df[zu_lang].groupby("groesse")["paper_id"].nunique()
        .reindex(uebersicht.index, fill_value=0)
    )
    uebersicht["kuerzester_zu_langer"] = (
        mess_df[zu_lang].groupby("groesse")["chars"].min().reindex(uebersicht.index).astype("Int64")
    )
    print(uebersicht.sort_index(ascending=False).to_string(float_format=lambda x: f"{x:.4f}"))

    beispiele = mess_df[zu_lang].sort_values("chars").head(5)
    if not beispiele.empty:
        print("\nKuerzeste zu lange Chunks:")
        for b in beispiele.itertuples(index=False):
            print(f"  {b.groesse} | {b.paper_id} | {b.chars} Zeichen: {b.anfang!r}")

Stichprobe: 10 Dokumente, 6312 Ollama-Aufrufe


600 Zeichen: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [05:21<00:00, 32.15s/it]

         chunks  zu_lang  fehler  anteil_zu_lang  dokumente_betroffen  kuerzester_zu_langer
groesse                                                                                    
1200       1309       22       0          0.0168                    6                   850
800        2078        4       0          0.0019                    1                   799
600        2925        0       0          0.0000                    0                  <NA>

Kuerzeste zu lange Chunks:
  800 | 2023_Jun_055 | 799 Zeichen: '|TgtLabel<br>100.0<br>100.0<br>100.0<br>100.0<br>100.0<br>100.0<br>FixedEpoch<br>N/A<br>N/A<br>N/A<br>N/A<br>N/A<br>N/A<'
  800 | 2023_Jun_055 | 799 Zeichen: '|TgtLabel<br>100.0<br>100.0<br>100.0<br>100.0<br>100.0<br>100.0<br>FixedEpoch<br>N/A<br>N/A<br>N/A<br>N/A<br>N/A<br>N/A<'
  800 | 2023_Jun_055 | 799 Zeichen: '|TgtLabel<br>100.0<br>100.0<br>100.0<br>100.0<br>100.0<br>100.0<br>FixedEpoch<br>N/A<br>N/A<br>N/A<br>N/A<br>N/A<br>N/A<'
  800 | 2023_Jun_055 | 799 Zeichen:

## 9. Removal of the reference list

**Thesis:** Document Preprocessing (removal of the reference list before chunking and
its validation).

Checks the cockpit function `entferne_literaturverzeichnis` on the full corpus: whether
the heading is found, how much text is removed, whether appendices are kept, and how
links are distributed between main text and reference list. The last point tests the
assumption that repository and dataset links in the reference list mostly belong to
other works. A very high removed share indicates a heading detected too early.

The two following cells check the cut points.

In [14]:
# Applies the reference list removal to all papers and reports found heading, removed share,
# kept appendices, and links in main text versus reference list.
MAX_ENTFERNT_ANTEIL = 0.5   # Darueber: Dokument manuell pruefen

LINK = re.compile(r"https?://\S+|github\.com/\S+", re.IGNORECASE)
REPO_LINK = re.compile(r"github\.com|gitlab\.com|bitbucket\.org|zenodo\.org|osf\.io|figshare\.com", re.IGNORECASE)


def lade_text(paper_id: str, jahr: str):
    """Volltext wie in extract_pdf_structure, aber vor dem Entfernen."""
    if DATEN_QUELLE == "markdown":
        pfad = MARKDOWN_ROOT / jahr / f"{paper_id}.md"
        return pfad.read_text(encoding="utf-8") if pfad.exists() else None
    pfad = STUDIEN_ROOT / jahr / f"{paper_id}.pdf"
    if not pfad.exists():
        return None
    with fitz.open(pfad) as doc:
        return "\n".join(page.get_text("text") for page in doc)


ref_report = []
for row in tqdm(master_label_df[["paper_id", "year"]].itertuples(index=False),
                total=len(master_label_df), desc="Literaturverzeichnis"):
    text = lade_text(row.paper_id, row.year)
    if text is None:
        ref_report.append({"paper_id": row.paper_id, "status": "quelle_fehlt"})
        continue
    rest, info = entferne_literaturverzeichnis(text, rohtext=(DATEN_QUELLE == "pdf_raw"))
    entfernt = ""
    if info["references_gefunden"]:
        entfernt = text[info["references_start"]:info["references_start"] + info["references_chars"]]
    ref_report.append({
        "paper_id": row.paper_id,
        "status": "ok" if info["references_gefunden"] else "keine_ueberschrift",
        **info,
        "entfernt_anteil": info["references_chars"] / len(text) if text else 0.0,
        "links_haupttext": len(LINK.findall(rest)),
        "links_references": len(LINK.findall(entfernt)),
        "repo_links_haupttext": len(REPO_LINK.findall(rest)),
        "repo_links_references": len(REPO_LINK.findall(entfernt)),
    })

ref_df = pd.DataFrame(ref_report)
ref_df.to_csv(VALIDATION_ROOT / f"references_report_{DATEN_QUELLE}.csv", index=False)

print(ref_df["status"].value_counts().to_string())
ok = ref_df[ref_df["status"] == "ok"]
print(f"\nAnhang nach dem Literaturverzeichnis erhalten: {int(ok['anhang_erhalten'].sum())} Dokumente")
print("Entfernter Anteil am Text:")
print(ok["entfernt_anteil"].describe(percentiles=[0.05, 0.5, 0.95]).to_string(float_format=lambda x: f"{x:.3f}"))

print("\nLinks gesamt (Haupttext / Literaturverzeichnis):")
print(f"  alle Links:       {int(ok['links_haupttext'].sum())} / {int(ok['links_references'].sum())}")
print(f"  Repository-Links: {int(ok['repo_links_haupttext'].sum())} / {int(ok['repo_links_references'].sum())}")

for titel, auswahl in [
    ("Ohne erkannte Ueberschrift", ref_df[ref_df["status"] == "keine_ueberschrift"]),
    (f"Entfernter Anteil ueber {MAX_ENTFERNT_ANTEIL:.0%}", ok[ok["entfernt_anteil"] > MAX_ENTFERNT_ANTEIL]),
]:
    if not auswahl.empty:
        print(f"\n{titel}: {len(auswahl)}")
        print(", ".join(auswahl["paper_id"].head(30)))

Literaturverzeichnis: 100%|█████████████████████████████████████████████████████████████████████████████████████| 654/654 [00:08<00:00, 73.82it/s]


status
ok    654

Anhang nach dem Literaturverzeichnis erhalten: 268 Dokumente
Entfernter Anteil am Text:
count   654.000
mean      0.177
std       0.052
min       0.073
5%        0.108
50%       0.169
95%       0.272
max       0.458

Links gesamt (Haupttext / Literaturverzeichnis):
  alle Links:       2380 / 13621
  Repository-Links: 285 / 175


In [15]:
# Plausibility of the cut points (Markdown only; requires the previous cell).
# A: lines detected as start of an appendix
# B: section-like headings inside the removed part (possible undetected appendices);
#    page headers and reference fragments set as headings by pymupdf4llm are ignored
# C: papers with the highest removed share, with paper type and number of references
UEBERSCHRIFT = re.compile(r"^[ \t]*#+[ \t]*(.+?)[ \t]*$", re.MULTILINE)
SEITENKOPF = re.compile(r"•|^\d+:\d+")
ABSCHNITT = re.compile(r"^(?:\*\*|_)?[ \t]*(?:(?:[A-Z]|\d+)(?:\.\d+)*\.?[ \t]+[A-Za-z]|[^\n]*\b(?i:appendix|appendices|supplementary)\b)")
EINTRAG = re.compile(r"^[ \t]*(?:-[ \t]*)?\[\d+\]", re.MULTILINE)

jahr_von = master_label_df.set_index("paper_id")["year"]
anhang_koepfe, verdacht, eintraege = [], [], {}

for row in ref_df[ref_df["status"] == "ok"].itertuples(index=False):
    text = lade_text(row.paper_id, jahr_von[row.paper_id])
    start, laenge = int(row.references_start), int(row.references_chars)
    eintraege[row.paper_id] = len(EINTRAG.findall(text[start:start + laenge]))
    if row.anhang_erhalten:
        anhang_koepfe.append((row.paper_id, text[start + laenge:].splitlines()[0].strip()))
    koepfe = [k for k in UEBERSCHRIFT.findall(text[start:start + laenge])[1:]
              if not SEITENKOPF.search(k) and ABSCHNITT.search(k)]
    if koepfe:
        verdacht.append({"paper_id": row.paper_id, "n_ueberschriften": len(koepfe),
                         "repo_links_references": row.repo_links_references,
                         "beispiele": " | ".join(koepfe[:3])})

anhang_df = pd.DataFrame(anhang_koepfe, columns=["paper_id", "anhang_ueberschrift"])
verdacht_df = pd.DataFrame(verdacht, columns=["paper_id", "n_ueberschriften", "repo_links_references", "beispiele"])
anhang_df.to_csv(VALIDATION_ROOT / f"references_anhaenge_{DATEN_QUELLE}.csv", index=False)
verdacht_df.to_csv(VALIDATION_ROOT / f"references_verdacht_{DATEN_QUELLE}.csv", index=False)

print(f"A: Erkannte Anhang-Ueberschriften: {len(anhang_df)} (Stichprobe)")
for r in anhang_df.sample(n=min(15, len(anhang_df)), random_state=GLOBAL_SEED).itertuples(index=False):
    print(f"  {r.paper_id}: {r.anhang_ueberschrift[:90]!r}")

print(f"\nB: Entfernte Teile mit weiteren Ueberschriften: {len(verdacht_df)}")
with pd.option_context("display.max_colwidth", 100, "display.width", 200):
    print(verdacht_df.sort_values("n_ueberschriften", ascending=False).head(20).to_string(index=False))

anteil_df = ref_df[ref_df["status"] == "ok"].nlargest(10, "entfernt_anteil").copy()
anteil_df["eintraege"] = anteil_df["paper_id"].map(eintraege)
anteil_df["paper_type"] = anteil_df["paper_id"].map(master_label_df.set_index("paper_id")["paper_type"])
print(f"\nC: Hoechster entfernter Anteil (Median Literatureintraege im Korpus: {pd.Series(eintraege).median():.0f})")
with pd.option_context("display.max_colwidth", 60, "display.width", 200):
    print(anteil_df[["paper_id", "entfernt_anteil", "references_chars", "eintraege", "paper_type"]]
          .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

A: Erkannte Anhang-Ueberschriften: 268 (Stichprobe)
  2024_Sep_114: '## A Interpolation'
  2025_Sep_142: '## A Motivation: Visualization of Stress Sensing Heterogeneity'
  2023_Jun_059: '## A STEP-BY-STEP DEMONSTRATION OF SPACE푋 MAG’S SPACE OPTIMIZATION'
  2024_Dec_168: '## A Prompt Details'
  2024_Sep_120: '## A DESCRIPTIVES'
  2025_Mar_021: '## A Details of Pointing-based Segmentation'
  2025_Dec_181: '## APPENDIX'
  2024_Dec_173: '## A Methodological Transparency & Reproducibility'
  2023_Sep_085: '## A DETAILED IMPLEMENTATION OF IK'
  2024_Mar_003: '## **Appendices**'
  2024_Dec_157: '## A Appendix: Network Communication Congestion'
  2024_Mar_040: '## A INDEX OF SUPPLEMENTARY MATERIAL'
  2024_Mar_033: '## A ANNOTATION DESIGN DETAILS'
  2025_Sep_083: '## A APPENDIX'
  2023_Jun_066: '## A ADDITIONAL USER STUDY DETAILS'

B: Entfernte Teile mit weiteren Ueberschriften: 0
Empty DataFrame
Columns: [paper_id, n_ueberschriften, repo_links_references, beispiele]
Index: []

C: Hoechster ent

In [16]:
# Markdown versus raw PDF text: appendices detected in Markdown but not in the raw text.
# Shows the raw text at the position of the Markdown heading to adjust the raw-text rule.
# Independent of DATEN_QUELLE.
N_BEISPIELE = 8


def rohtext(paper_id: str, jahr: str) -> str:
    with fitz.open(STUDIEN_ROOT / jahr / f"{paper_id}.pdf") as doc:
        return "\n".join(page.get_text("text") for page in doc)


fitz.TOOLS.mupdf_display_errors(False)
abweichend = []
for row in tqdm(master_label_df[["paper_id", "year"]].itertuples(index=False),
                total=len(master_label_df), desc="Markdown vs. Rohtext"):
    md_pfad = MARKDOWN_ROOT / row.year / f"{row.paper_id}.md"
    if not md_pfad.exists():
        continue
    md = md_pfad.read_text(encoding="utf-8")
    _, md_info = entferne_literaturverzeichnis(md)
    if not md_info["anhang_erhalten"]:
        continue
    roh = rohtext(row.paper_id, row.year)
    _, roh_info = entferne_literaturverzeichnis(roh, rohtext=True)
    if not roh_info["anhang_erhalten"]:
        kopf = md[md_info["references_start"] + md_info["references_chars"]:].splitlines()[0]
        abweichend.append((row.paper_id, kopf, roh))

print(f"Anhang im Markdown erkannt, im Rohtext nicht: {len(abweichend)}")
for p_id, kopf, roh in abweichend[:N_BEISPIELE]:
    titel = re.sub(r"[#*_]", "", kopf).strip()
    wort = max(titel.split(), key=len) if titel.split() else titel
    pos = roh.lower().rfind(wort.lower())
    print(f"\n{p_id} | Markdown: {kopf.strip()[:80]!r}")
    if pos < 0:
        print(f"  Suchwort {wort!r} im Rohtext nicht gefunden")
        continue
    zeilen_start = roh.rfind("\n", 0, max(0, pos - 150)) + 1
    for zeile in roh[zeilen_start:pos + 200].splitlines()[:8]:
        print(f"  | {zeile!r}")

Markdown vs. Rohtext: 100%|█████████████████████████████████████████████████████████████████████████████████████| 654/654 [00:43<00:00, 14.99it/s]

Anhang im Markdown erkannt, im Rohtext nicht: 1

2025_Dec_172 | Markdown: '## A SEARCH DETAILS CORRESPONDING TO EACH SOURCE'
  | 'Proc. ACM Interact. Mob. Wearable Ubiquitous Technol., Vol. 9, No. 4, Article 172. Publication date: December 2025.'
  | ''
  | '172:40'
  | '•'
  | 'Fang et al.'
  | 'A'
  | 'SEARCH DETAILS CORRESPONDING TO EACH SOURCE'
  | 'ACM Digital Library'


## 10. Inventory of embedding and fusion folders

**Thesis:** no direct statement; documents which stored embeddings and fusion data
exist and with which configuration they were created, so that every reported result
can be traced to its inputs.

Folder names follow data source, model, and `STRATEGIE_NAME` (chunk size, overlap,
`_prefixed` for the document prefix, `_norefs` for the removed reference list).
Folders without a manifest stem from versions before the embedding manifest.

In [17]:
# Lists all embedding and fusion folders with number of files, manifest, creation time,
# and the configuration stored in the embeddings (prefix, reference removal, splitting).
def embedding_ordner_info(ordner: Path) -> dict:
    dateien = sorted(ordner.rglob("*_embeddings.json"))
    info = {
        "ordner": str(ordner.relative_to(EMBEDDING_BASE_ROOT)),
        "aktuell": ordner == CURRENT_EMBEDDING_PATH,
        "json_dateien": len(dateien),
        "manifest": (ordner / "embedding_manifest.csv").exists(),
        "lauf_abgebrochen": (ordner / "embedding_lauf.json").exists(),
        "erstellt_von": None, "erstellt_bis": None, "konfiguration": None,
    }
    if dateien:
        zeiten = [d.stat().st_mtime for d in dateien]
        info["erstellt_von"] = datetime.fromtimestamp(min(zeiten)).strftime("%Y-%m-%d %H:%M")
        info["erstellt_bis"] = datetime.fromtimestamp(max(zeiten)).strftime("%Y-%m-%d %H:%M")
        meta = json.loads(dateien[0].read_text(encoding="utf-8")).get("metadata", {})
        cfg = meta.get("embed_config")
        info["konfiguration"] = (
            f"prefix={cfg.get('prefix')!r}, refs_entfernt={cfg.get('references_entfernen', False)}, "
            f"nachteilen={'nachteilen' in cfg}" if cfg else "alt (ohne gespeicherte Konfiguration)"
        )
    if info["manifest"]:
        m = pd.read_csv(ordner / "embedding_manifest.csv")
        info["manifest_studien"] = len(m)
        info["konfigurationen_im_manifest"] = m["config_sha256"].nunique()
    return info


# Ordnerebene: <quelle>/<modell>/<strategie>
emb_ordner = [p for p in EMBEDDING_BASE_ROOT.glob("*/*/*") if p.is_dir()]
emb_df = pd.DataFrame([embedding_ordner_info(p) for p in sorted(emb_ordner)])
with pd.option_context("display.max_colwidth", 80, "display.width", 250):
    print(emb_df.to_string(index=False))

fus = []
for ordner in sorted(p for p in FUSION_BASE_ROOT.glob("*/*/*/*") if p.is_dir()):
    x_pfad = ordner / "X_data.npy"
    fus.append({
        "ordner": str(ordner.relative_to(FUSION_BASE_ROOT)),
        "X_vorhanden": x_pfad.exists(),
        "manifest": (ordner / "fusion_manifest.json").exists(),
        "erstellt": datetime.fromtimestamp(x_pfad.stat().st_mtime).strftime("%Y-%m-%d %H:%M") if x_pfad.exists() else None,
    })
print()
with pd.option_context("display.max_colwidth", 90, "display.width", 250):
    print(pd.DataFrame(fus).to_string(index=False))

                                                               ordner  aktuell  json_dateien  manifest  lauf_abgebrochen     erstellt_von     erstellt_bis                                                    konfiguration  manifest_studien  konfigurationen_im_manifest
                   markdown/mxbai-embed-large/1200_chars_fixed_chunks    False           654     False             False 2026-06-02 16:40 2026-06-02 17:48                            alt (ohne gespeicherte Konfiguration)               NaN                          NaN
               markdown/mxbai-embed-large/recursive_1200c_200o_norefs    False           654      True             False 2026-10-09 15:35 2026-10-09 16:29                   prefix='', refs_entfernt=True, nachteilen=True             654.0                          1.0
             markdown/nomic-embed-text-v2-moe/1200_chars_fixed_chunks    False           654     False             False 2026-06-09 20:57 2026-06-10 08:29                            alt (ohne gespeic

NameError: name 'BASE_DIR' is not defined